# 01. Multi-Turn Conversations With a Foundry Agent

This notebook opens **AI-103 -> Section 03: AI Foundry agent conversations & evaluation**. Where Section 01 pointed the plain `openai` SDK straight at an Azure OpenAI *resource*, this one goes through `azure.ai.projects.AIProjectClient` to reach an Azure AI Foundry **project**, opens a server-managed `conversation`, and routes each turn to a **named Agent** (`cloudxeus-support-agent-conv`) that was created ahead of time in the Foundry portal -- see the companion no-code lab `03. Section Code/02. Lab - Creating the CloudXeus Agents/` for how that agent itself gets built. This continues the course's running "CloudXeus" customer-support scenario.

**Difficulty: Intermediate**


## Prerequisites

**Python packages** (already in this repo's root `requirements.txt`):
- `azure-ai-projects` (`AIProjectClient`)
- `azure-identity` (`DefaultAzureCredential`)
- `python-dotenv`

**Azure resources**
- An Azure AI Foundry **project** (a higher-level container than a plain Azure OpenAI resource -- see the Exam tip below) with an **Agent already created and named** to match `AGENT_NAME` (built via the Foundry portal; see the companion lab `02. Lab - Creating the CloudXeus Agents/` in this same folder for the no-code steps).
- Entra ID auth via `az login`, with an RBAC role on the project such as **Azure AI Developer**.

**Environment variables** -- this repo's root `.env` already defines `AZURE_AI_PROJECT_ENDPOINT` for the labs in `11_azure_ai_foundry/`; this notebook reuses that same variable. Add, if not already present:
```
AZURE_AI_PROJECT_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/api/projects/<your-project>
AZURE_AI_AGENT_NAME=cloudxeus-support-agent-conv
```
> `AZURE_AI_AGENT_NAME` defaults to `cloudxeus-support-agent-conv` (the original script's hardcoded value) if you don't set it.


## What You'll Learn
- `AIProjectClient` as the SDK's project-level entry point (vs. a per-resource OpenAI client)
- `get_openai_client()` -- bridging a Foundry project into the familiar OpenAI SDK surface
- `conversations.create()` -- server-managed multi-turn state you don't have to rebuild yourself
- The `extra_body={"agent_reference": {...}}` pattern for routing a call to a named Agent
- Companion no-code labs: this section also ships portal-only exercises for creating the agent (`02. Lab - ...`), capturing structured output (`03. Lab - ...`), and adding workflow conditions (`04. Lab - ...`) -- worth exploring in the Foundry portal alongside this notebook.


### 1. Imports, configuration, and the project client

`AIProjectClient(endpoint, credential)` is the top-level handle to your Foundry **project** -- the container that can host multiple agents, data connections, and model deployments. `DefaultAzureCredential()` reuses your `az login` session, same as the Entra ID pattern in Section 01.

**Exam tip:** Keep the vocabulary straight for AI-102: an **Azure OpenAI resource** (used directly in Section 01's notebooks) is a single Cognitive Services resource with model deployments; an **Azure AI Foundry project** (used here) is a higher-level workspace that can group multiple agents, connections to external data/tools, and deployments together. `AIProjectClient` is the SDK entry point for project-level operations, distinct from a plain `OpenAI`/`AzureOpenAI` client pointed at one resource.


In [ ]:
import os

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv()

PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
AGENT_NAME = os.getenv("AZURE_AI_AGENT_NAME", "cloudxeus-support-agent-conv")

client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)


### 2. Bridge to the OpenAI SDK and open a conversation

`get_openai_client()` returns a standard OpenAI-SDK-shaped client that's already pointed at this project's endpoint/auth. `openai_client.conversations.create()` creates a server-side conversation object; we'll reuse its `id` across calls so we only ever send the *new* turn, not the whole history. "Sara" here just stands in for one customer's support conversation in the CloudXeus scenario.

**Exam tip:** This `conversations` surface is the OpenAI-compatible API a Foundry project exposes via `get_openai_client()`. It's conceptually similar to -- but a different object model from -- the `azure-ai-agents` SDK's own `Thread` object (used for the persistent Agent Service pattern in `11_azure_ai_foundry/05_hosted_agent/` in this repo). Both give you server-side multi-turn memory; which one you reach for depends on which SDK/API surface the rest of your app already speaks.


In [ ]:
# Sara's conversation -- server-side state, so we only send the new turn each time
openai_client = client.get_openai_client()
sara_conversation = openai_client.conversations.create()
print(sara_conversation.id)


### 3. First turn -- route the request to the named Agent

`conversation=sara_conversation.id` ties this call into the conversation we just opened. `extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}}` routes the call to the pre-created, portal-declared Agent instead of a bare model deployment -- so the Agent's own instructions, tools, and knowledge sources apply.

**Exam tip:** `extra_body` is the `openai` Python SDK's escape hatch for sending fields the base client doesn't have typed support for -- Azure/Foundry-specific extensions like `agent_reference` ride through it. This is a useful thing to recognize on the exam: the installed SDK (`openai`) is generic, and Azure-specific routing/behavior is layered on through `base_url` plus `extra_body`/headers, not through SDK code changes.


In [ ]:
response = openai_client.responses.create(
    conversation=sara_conversation.id,
    extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}},
    input="My order #4521 is late.",
)

print(response.output_text)


### 4. Second turn -- same conversation, no need to repeat context

This call reuses the same `sara_conversation.id` and sends only the new message, "Any update on it?". The agent can resolve "it" because the conversation object retains the prior turn server-side -- we never resend the order number ourselves.

**Exam tip:** This is the core benefit of server-managed conversation state over manually building up an `input` array yourself on every call: less client-side bookkeeping, and it composes with a Foundry project's own conversation/thread storage and retention settings.


In [ ]:
response = openai_client.responses.create(
    conversation=sara_conversation.id,
    extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}},
    input="Any update on it?",
)

print(response.output_text)


## Summary

You created a project client, opened a server-side conversation, and sent two turns routed to a named Foundry Agent -- the second turn's pronoun ("it") was resolved correctly purely because the conversation object retained memory of the first turn.


## Try It Yourself
1. **Easy:** Add a third turn asking something unrelated to the order, and see how the agent handles the topic switch.
2. **Medium:** Start a second, independent `conversations.create()` and show it has no memory of Sara's order -- confirming conversation state is scoped per conversation id, not global.
3. **Advanced:** Wrap the repeated call pattern into a small `chat(conversation_id, message)` helper function, then build a tiny REPL loop around it for interactive testing.
